# SolveBench — Solving $A\vec{x}=\vec{b}$ at Scale

### Direct vs. Iterative Linear Solvers on Real Engineering Data

**CSE 402 — Numerical Analysis Lab Project** · Section A2
Roll Numbers: 2105037 · 2105038 · 2105042 · 2105047 · 2105054

---

## What this notebook does

Every real engineering simulation — modelling a power grid, checking a structure's
safety, or simulating a circuit — comes down to the same core step: solving
$A\vec{x}=\vec{b}$. Get the solver choice wrong, and what should take milliseconds
either **crawls** or **never converges**.

This notebook benchmarks **8 classical solvers** against **49 real engineering
matrices** drawn from the SuiteSparse Matrix Collection, spanning **3 domains**.

| | |
|---|---|
| **Base paper** | P. Khrapov & N. Volkov (2024), *Comparative Analysis of Jacobi and Gauss-Seidel Iterative Methods*, International Journal of Open Information Technologies, 12(2) |
| **The gap we fill** | The paper stops at **5 unknowns**, using only randomly generated matrices, and never measures speed. We test at **up to 9,774 unknowns** on **real** data, and measure runtime, iterations, residual, error, and conditioning. |
| **Category** | Exploratory & Comparative Study |

### The 8 solvers

| Direct (exact, fixed number of steps) | Iterative (refine a guess, may not converge) |
|---|---|
| Gauss elimination | Jacobi |
| Gauss-Jordan | Gauss-Seidel |
| LU decomposition | SOR (Successive Over-Relaxation) |
| Cholesky | Conjugate Gradient (CG) |

### Ground-truth method

For each matrix $A$ we **construct** a known answer, so "correct" is never ambiguous:

$$\vec{x}_{\text{true}} = (1,1,\dots,1)^T \qquad \vec{b} = A\vec{x}_{\text{true}}$$

then solve $A\vec{x}=\vec{b}$ with every method and compare against $\vec{x}_{\text{true}}$.
Fully deterministic — no randomness anywhere in the pipeline.

---
# 1 · Configuration & Environment

Everything tunable lives in this one cell.

In [ ]:
import os, sys, time, json, math, warnings, platform, textwrap, shutil
from pathlib import Path
from datetime import datetime

# ----------------------------------------------------------------- CONFIG
CONFIG = {
    # Iterative solver settings
    "max_iterations":  10_000,   # cap for iterative methods before declaring non-convergence
    "tolerance":       1e-8,     # relative residual tolerance ||Ax-b||/||b||
    "sor_omega":       1.25,
    "refinement_passes": 1,      # APK refinement passes; measured: 1 pass gives the
                                 # full ~215,000x accuracy gain, a 2nd adds nothing     # SOR relaxation factor (1.0 == Gauss-Seidel)

    # Our direct solvers are hand-written (no LAPACK), so they cost O(n^3) in
    # Python and get impractical past a few thousand unknowns. Iterative solvers
    # are sparse and stay fast at any size, so only DIRECT methods are capped.
    "direct_size_cap": 2000,

    # Exact condition number via SVD is O(n^3); above this we use a cheaper
    # 1-norm estimator (splu + onenormest) so every matrix still gets a number.
    "cond_exact_cap":  2000,
}

# ----------------------------------------------------------------- PATHS
ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:
    # Search at any depth; Kaggle mount layout can vary between datasets.
    _candidates = sorted(Path("/kaggle/input").rglob("manifest.csv"))
    if not _candidates:
        # Print what IS mounted, so a failure here is diagnosable at a glance
        print("!! manifest.csv not found. Contents of /kaggle/input:")
        _any = False
        for _p in sorted(Path("/kaggle/input").rglob("*"))[:60]:
            print("   ", _p); _any = True
        if not _any:
            print("    (empty — the dataset is attached in metadata but not mounted yet;")
            print("     this usually means it was still processing when the run started)")
        raise FileNotFoundError("Could not find manifest.csv under /kaggle/input")
    DATASET_ROOT = _candidates[0].parent
    OUTPUT_ROOT  = Path("/kaggle/working/solvebench_output")
else:
    DATASET_ROOT = Path("dataset_large")
    OUTPUT_ROOT  = Path("results/solvebench_output")

FIG_DIR    = OUTPUT_ROOT / "figures"
TABLE_DIR  = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
LOG_DIR    = OUTPUT_ROOT / "logs"
for _d in (FIG_DIR, TABLE_DIR, REPORT_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.now()

print("=" * 78)
print("SOLVEBENCH — configuration")
print("=" * 78)
print(f"  Running on Kaggle : {ON_KAGGLE}")
print(f"  Python            : {sys.version.split()[0]} ({platform.system()})")
print(f"  Dataset root      : {DATASET_ROOT.resolve()}")
print(f"  Output root       : {OUTPUT_ROOT.resolve()}")
print(f"  Run started       : {RUN_STARTED:%Y-%m-%d %H:%M:%S}")
print("-" * 78)
for k, v in CONFIG.items():
    print(f"  {k:<18}: {v}")
print("=" * 78)

In [ ]:
import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

print("Library versions")
print(f"  numpy      {np.__version__}")
print(f"  scipy      {__import__('scipy').__version__}")
print(f"  pandas     {pd.__version__}")
print(f"  matplotlib {matplotlib.__version__}")

# CPU info — this workload is pure CPU linear algebra, no GPU involved
try:
    print(f"  CPU cores  {os.cpu_count()}")
except Exception:
    pass

---
# 2 · Visual identity

Chart colours are inherited from the project's own slide deck (`linear.tex`), then
**adjusted until they pass an automated accessibility audit** — colour-blind
separation, chroma, lightness band, and contrast against the page.

Because 8 distinct hues cannot pass that audit, the plots use **composite
encoding** instead of 8 colours:

* **Colour** carries the meaningful split — <span style="color:#1596A5">**teal = direct**</span>, <span style="color:#C4453F">**red = iterative**</span>
* **Marker shape** identifies the individual method

That is not a workaround: direct-vs-iterative *is* the project's central question,
so the strongest visual channel encodes it.

In [ ]:
# ---- validated categorical palette (audited for CVD separation & contrast) ----
C_TEAL, C_RED, C_GOLD, C_BLUE = "#1596A5", "#C4453F", "#B8862B", "#2E6FB7"
C_GREEN = "#2F8F5B"

# ink / surface tokens (from linear.tex)
INK, INK_SOFT, INK_MUTED = "#0F3450", "#3C4C5A", "#5C646C"
SURFACE, GRIDLINE = "#FCFBF8", "#DCE3E6"

# The benchmark now spans many domains, so colours are assigned from a palette
# once the manifest is known (see the dataset cell) rather than hardcoded.
DOMAIN_PALETTE = [C_TEAL, C_RED, C_GOLD, C_GREEN, "#7B5EA7", "#B5651D", "#2F6F6F",
                  "#8A3B5F", "#4C6EB1", "#93761F", "#3F7D5A", "#A0522D"]
DOMAIN_COLORS = {}
def domain_color(d):
    return DOMAIN_COLORS.get(d, INK_MUTED)
FAMILY_COLORS = {"direct": C_TEAL, "iterative": C_RED}

DIRECT_METHODS_ORDER    = ["Gauss elimination", "Gauss-Jordan", "LU", "Cholesky"]
ITERATIVE_METHODS_ORDER = ["Jacobi", "Gauss-Seidel", "SOR", "Conjugate Gradient",
                           "BiCGSTAB", "APK (ours)"]
METHOD_ORDER = DIRECT_METHODS_ORDER + ITERATIVE_METHODS_ORDER

# marker identifies the specific method within its family
METHOD_MARKERS = {
    "Gauss elimination": "o", "Gauss-Jordan": "s", "LU": "^", "Cholesky": "D",
    "Jacobi": "o", "Gauss-Seidel": "s", "SOR": "^", "Conjugate Gradient": "D",
    "BiCGSTAB": "v", "APK (ours)": "*",
}
METHOD_FAMILY = ({m: "direct" for m in DIRECT_METHODS_ORDER} |
                 {m: "iterative" for m in ITERATIVE_METHODS_ORDER})

# single-hue sequential ramp for magnitude heatmaps (never a rainbow)
SEQ_CMAP = LinearSegmentedColormap.from_list("solvebench_seq", ["#F2FAFB", "#9FD5DC", "#1596A5", "#0B4E57"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRIDLINE, "axes.labelcolor": INK_SOFT, "axes.titlecolor": INK,
    "text.color": INK, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "grid.color": GRIDLINE, "grid.linewidth": 0.8, "axes.grid": True, "grid.alpha": 0.7,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlepad": 12, "legend.frameon": False, "figure.dpi": 110,
})

SAVED_FIGURES = []
def save_fig(fig, name, caption=""):
    """Save a figure to the output folder and register it for the final report."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    SAVED_FIGURES.append({"name": name, "file": path.name, "caption": caption})
    print(f"  [figure saved] {path.name}")
    return path

print("Palette locked in and audited:")
print(f"  direct    -> {C_TEAL}")
print(f"  iterative -> {C_RED}")
print(f"  palette   -> {len(DOMAIN_PALETTE)} colours, assigned per domain once the manifest loads")

---
# 3 · The pipeline

This reproduces the methodology diagram from the project proposal, so the notebook
and the slide deck tell exactly the same story.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.6))
ax.set_xlim(0, 100); ax.set_ylim(0, 30); ax.axis("off"); ax.grid(False)

def box(x, y, w, h, title, sub, edge, fill, bold=True):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.6,rounding_size=1.2",
                                 linewidth=1.6, edgecolor=edge, facecolor=fill))
    ax.text(x + w/2, y + h/2 + (1.6 if sub else 0), title, ha="center", va="center",
            fontsize=9.5, fontweight="bold" if bold else "normal", color=INK)
    if sub:
        ax.text(x + w/2, y + h/2 - 2.6, sub, ha="center", va="center",
                fontsize=7.3, color=INK_MUTED)

def arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=13,
                                  linewidth=1.4, color=C_TEAL, shrinkA=0, shrinkB=0))

box(1,  11, 17, 9, "Real sparse matrix", "SuiteSparse .mtx", C_GOLD, "#FBF6EC")
box(26, 19, 24, 9, "Direct solvers", "Gauss Elim. / Gauss-Jordan / LU / Cholesky", C_TEAL, "#EAF4F6")
box(26,  3, 24, 9, "Iterative solvers", "Jacobi / Gauss-Seidel / SOR / CG", C_TEAL, "#EAF4F6")
box(58, 11, 22, 9, "Benchmark harness", "runtime · iterations · residual\nerror · condition no.", C_TEAL, "#EAF4F6")
box(84, 11, 15, 9, "Results dashboard", "heatmaps · plots\ndecision guide", C_GREEN, "#EDF6F1")

arrow(18.5, 17, 25.5, 22); arrow(18.5, 14, 25.5, 9)
arrow(50.5, 22, 57.5, 17); arrow(50.5, 9, 57.5, 14)
arrow(80.5, 15.5, 83.5, 15.5)

ax.set_title("SolveBench methodology pipeline", fontsize=13, pad=6, loc="left")
save_fig(fig, "00_pipeline_diagram", "The benchmark pipeline, mirroring the project proposal.")
plt.show()

---
# 4 · The dataset

49 real matrices from the [SuiteSparse Matrix Collection](https://sparse.tamu.edu/),
curated into three engineering domains. Every matrix is **square** and stores **real
numeric values** (no structure-only "pattern" files), so all of them are usable
without synthetic preprocessing.

| Domain | What the matrix physically represents |
|---|---|
| **Power network** | Bus admittance — how current relates to voltage across a grid's transmission lines. Solving $A\vec{x}=\vec{b}$ here is what a power-flow study computes. |
| **Structural** | Stiffness / mass — how much force produces a given displacement in a finite-element model of a building or bridge. |
| **Circuit simulation** | Node equations solved at every timestep of a transient circuit simulation (the matrix inside a SPICE-style tool). |

In [ ]:
manifest = pd.read_csv(DATASET_ROOT / "manifest.csv")

# assign a stable colour to every domain present
DOMAIN_COLORS.update({d: DOMAIN_PALETTE[i % len(DOMAIN_PALETTE)]
                      for i, d in enumerate(sorted(manifest["domain"].unique()))})

def discover_matrices(root: Path):
    """Walk <root>/<domain>/<name>.mtx."""
    found = []
    for domain_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        for mtx in sorted(domain_dir.glob("*.mtx")):
            found.append({"domain": domain_dir.name, "name": mtx.stem, "path": mtx})
    return found

MATRICES = discover_matrices(DATASET_ROOT)

print(f"Discovered {len(MATRICES)} matrices on disk\n")
summary = (manifest.groupby("domain")
           .agg(matrices=("name", "count"), smallest=("rows", "min"),
                largest=("rows", "max"), total_nonzeros=("nnz", "sum"))
           .reset_index())
print(summary.to_string(index=False))
print(f"\n  TOTAL: {len(manifest)} matrices | "
      f"n from {manifest['rows'].min():,} to {manifest['rows'].max():,} | "
      f"{manifest['nnz'].sum():,} stored nonzeros")

n_capped = int((manifest["rows"] > CONFIG["direct_size_cap"]).sum())
print(f"\n  Direct-solver size cap = {CONFIG['direct_size_cap']:,}")
print(f"    {len(manifest) - n_capped} matrices get all 8 methods")
print(f"    {n_capped} matrices get the 4 iterative methods only (too large for hand-written O(n^3) code)")

manifest.to_csv(TABLE_DIR / "01_dataset_manifest.csv", index=False)
summary.to_csv(TABLE_DIR / "02_dataset_summary_by_domain.csv", index=False)
print(f"\n  [table saved] 01_dataset_manifest.csv, 02_dataset_summary_by_domain.csv")

In [ ]:
# ---------------------------------------------- FIGURE: dataset composition
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# (a) matrices per domain
ax = axes[0]
doms = summary.sort_values("matrices", ascending=True)
colors = [domain_color(d) for d in doms["domain"]]
bars = ax.barh(doms["domain"].str.replace("_", " ").str.title(), doms["matrices"],
               color=colors, height=0.55)
for bar, val in zip(bars, doms["matrices"]):
    ax.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height()/2, str(val),
            va="center", fontsize=10, fontweight="bold", color=INK)
ax.set_xlabel("number of matrices"); ax.set_title("Matrices per domain", loc="left")
ax.set_xlim(0, doms["matrices"].max() * 1.18); ax.grid(axis="y", visible=False)

# (b) size distribution
ax = axes[1]
for dom, grp in manifest.groupby("domain"):
    ax.scatter(grp["rows"], grp["nnz"], s=42, color=domain_color(dom),
               edgecolor=SURFACE, linewidth=1.2, alpha=0.9,
               label=dom.replace("_", " ").title())
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("matrix size $n$ (unknowns)"); ax.set_ylabel("stored nonzeros")
ax.set_title("Size vs. sparsity of every matrix", loc="left")
ax.legend(fontsize=8.5, loc="upper left")
ax.axvline(CONFIG["direct_size_cap"], color=INK_MUTED, linestyle="--", linewidth=1.1, alpha=0.8)
ax.text(CONFIG["direct_size_cap"] * 1.06, ax.get_ylim()[0] * 2.2,
        f"direct-solver cap\nn = {CONFIG['direct_size_cap']:,}", fontsize=7.5, color=INK_MUTED)

fig.suptitle(f"The {len(manifest)}-matrix benchmark set", fontsize=14, fontweight="bold",
             x=0.005, ha="left", y=1.04, color=INK)
fig.tight_layout()
save_fig(fig, "01_dataset_composition", "Composition and size/sparsity spread of the benchmark set.")
plt.show()

In [ ]:
# ---------------------------------------------- FIGURE: sparsity patterns
picks = []
_top_domains = (manifest.groupby("domain").size().sort_values(ascending=False)
                .head(4).index.tolist())
for dom in _top_domains:
    sub = manifest[manifest["domain"] == dom].sort_values("rows")
    if len(sub):
        picks.append((dom, sub.iloc[len(sub)//2]["name"]))

fig, axes = plt.subplots(1, len(picks), figsize=(4.3 * len(picks), 4.6))
axes = np.atleast_1d(axes)
print("Rendering sparsity patterns (the 'shape' of each domain's physics):")
for ax, (dom, name) in zip(axes, picks):
    A = sp.csr_matrix(scipy.io.mmread(str(DATASET_ROOT / dom / f"{name}.mtx")))
    ax.spy(A, markersize=0.35, color=domain_color(dom))
    ax.set_title(f"{name}\n{dom.replace('_',' ').title()} · n={A.shape[0]:,} · nnz={A.nnz:,}",
                 fontsize=9.5, loc="left")
    ax.grid(False); ax.tick_params(labelsize=7)
    print(f"  {dom:<20s} {name:<16s} n={A.shape[0]:,}  nnz={A.nnz:,}  "
          f"density={A.nnz/A.shape[0]**2:.2e}")

fig.suptitle("Sparsity patterns — each domain has a distinct structural signature",
             fontsize=13, fontweight="bold", x=0.005, ha="left", y=1.02, color=INK)
fig.tight_layout()
save_fig(fig, "02_sparsity_patterns", "Nonzero structure of a representative matrix from each domain.")
plt.show()

---
# 5 · Ground truth

The trick that makes every measurement unambiguous: rather than hunting for a matrix
whose answer we happen to know, we **choose** the answer first and derive the
right-hand side from it.

$$\vec{x}_{\text{true}} = (1,1,\dots,1)^T, \qquad \vec{b} := A\,\vec{x}_{\text{true}}$$

Now $A\vec{x}=\vec{b}$ has a solution we already know exactly, so every solver can be
scored two independent ways:

| Metric | Formula | What it measures |
|---|---|---|
| **Relative residual** | $\dfrac{\lVert A\vec{x}_{\text{found}}-\vec{b}\rVert}{\lVert \vec{b}\rVert}$ | how well the *equation* is satisfied |
| **Relative error** | $\dfrac{\lVert \vec{x}_{\text{found}}-\vec{x}_{\text{true}}\rVert}{\lVert \vec{x}_{\text{true}}\rVert}$ | how close the *answer* actually is |

These are **not** the same thing — on an ill-conditioned matrix a solver can have a
tiny residual while still being far from the true answer. Reporting both is what
lets the condition-number analysis later in this notebook mean anything.

> Both are reported **relative** (normalised) because matrix entries in this dataset
> span roughly `3e3` to `2.5e9` — raw absolute norms would make well-solved large-valued
> matrices look worse than badly-solved small-valued ones.

In [ ]:
def load_matrix(mtx_path: Path) -> sp.csr_matrix:
    A = sp.csr_matrix(scipy.io.mmread(str(mtx_path)), dtype=np.float64)
    if A.shape[0] != A.shape[1]:
        raise ValueError(f"not square: {A.shape}")
    return A

def make_ground_truth(A: sp.csr_matrix):
    x_true = np.ones(A.shape[0], dtype=np.float64)
    return x_true, A @ x_true

# quick demonstration on the smallest matrix in the set
_demo_name = manifest.loc[manifest["rows"].idxmin()]
_demo_path = DATASET_ROOT / _demo_name["domain"] / (_demo_name["name"] + ".mtx")
_A = load_matrix(_demo_path)
_x, _b = make_ground_truth(_A)
print(f"Ground-truth demo on '{_demo_name['name']}'  (n={_A.shape[0]})")
print(f"  x_true = [1, 1, ..., 1]        ||x_true|| = {np.linalg.norm(_x):.4f}")
print(f"  b = A @ x_true                 ||b||      = {np.linalg.norm(_b):.4e}")
print(f"  sanity check ||A@x_true - b||  = {np.linalg.norm(_A @ _x - _b):.2e}  (must be 0)")

---
# 6 · The four direct solvers

Direct methods run a **fixed, predictable sequence** of arithmetic and land on the
exact answer (up to floating-point rounding). They never "fail to converge" — but
they cost $O(n^3)$, which is why they are capped by size here.

Every one of these is written from scratch. `numpy` is used only for vectorised row
arithmetic, never for the algorithm itself (no `np.linalg.solve` anywhere).

**Step counting.** Direct methods have no notion of "iterating until close enough",
but they *do* have a well-defined stage count — the number of elimination /
factorisation passes. We record it so both families can be compared on the same axis:

| Method | Stages |
|---|---|
| Gauss elimination, LU | $n-1$ (one per eliminated column) |
| Gauss-Jordan, Cholesky | $n$ (one per row) |

That contrast is itself a finding: for direct methods this count is **fixed by $n$
alone**, while for iterative methods it depends on the matrix's numerical behaviour.

In [ ]:
class SolverNotApplicable(Exception):
    """The matrix does not meet this method's mathematical requirements."""

def _partial_pivot(A, b, k):
    """Swap in the largest-magnitude pivot from the rows below — keeps elimination stable."""
    p = k + int(np.argmax(np.abs(A[k:, k])))
    if abs(A[p, k]) < 1e-14:
        raise SolverNotApplicable("singular (zero pivot even after partial pivoting)")
    if p != k:
        A[[k, p]] = A[[p, k]]
        b[[k, p]] = b[[p, k]]

def gauss_elimination(A_dense, b):
    """Forward-eliminate to upper triangular, then back-substitute."""
    n = A_dense.shape[0]; A = A_dense.copy(); b = b.copy(); steps = 0
    for k in range(n - 1):
        _partial_pivot(A, b, k)
        f = A[k+1:, k] / A[k, k]
        A[k+1:, k:] -= np.outer(f, A[k, k:])
        b[k+1:]     -= f * b[k]
        steps += 1
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - A[i, i+1:] @ x[i+1:]) / A[i, i]
    return x, steps

def gauss_jordan(A_dense, b):
    """Reduce all the way to the identity — the answer falls out, no back-substitution."""
    n = A_dense.shape[0]; A = A_dense.copy(); b = b.copy(); steps = 0
    for k in range(n):
        _partial_pivot(A, b, k)
        piv = A[k, k]
        A[k, :] /= piv; b[k] /= piv
        col = A[:, k].copy(); col[k] = 0.0
        A -= np.outer(col, A[k, :])
        b -= col * b[k]
        steps += 1
    return b, steps

def lu_solve(A_dense, b):
    """Doolittle LU with partial pivoting, then forward + back substitution."""
    n = A_dense.shape[0]; U = A_dense.copy(); L = np.eye(n); pb = b.copy(); steps = 0
    for k in range(n - 1):
        p = k + int(np.argmax(np.abs(U[k:, k])))
        if abs(U[p, k]) < 1e-14:
            raise SolverNotApplicable("singular (zero pivot even after partial pivoting)")
        if p != k:
            U[[k, p]] = U[[p, k]]; pb[[k, p]] = pb[[p, k]]
            if k > 0:
                L[[k, p], :k] = L[[p, k], :k]
        f = U[k+1:, k] / U[k, k]
        L[k+1:, k] = f
        U[k+1:, k:] -= np.outer(f, U[k, k:])
        steps += 1
    y = np.zeros(n)
    for i in range(n):
        y[i] = pb[i] - L[i, :i] @ y[:i]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x, steps

def cholesky_solve(A_dense, b, sym_tol=1e-8):
    """A = R^T R — only valid for symmetric positive-definite matrices."""
    n = A_dense.shape[0]
    if not np.allclose(A_dense, A_dense.T, atol=sym_tol, rtol=sym_tol):
        raise SolverNotApplicable("not symmetric")
    R = np.zeros((n, n)); steps = 0
    for i in range(n):
        d = A_dense[i, i] - R[:i, i] @ R[:i, i]
        if d <= 0:
            raise SolverNotApplicable("not positive-definite")
        R[i, i] = np.sqrt(d)
        if i + 1 < n:
            R[i, i+1:] = (A_dense[i, i+1:] - R[:i, i] @ R[:i, i+1:]) / R[i, i]
        steps += 1
    y = np.zeros(n)
    for i in range(n):
        y[i] = (b[i] - R[:i, i] @ y[:i]) / R[i, i]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - R[i, i+1:] @ x[i+1:]) / R[i, i]
    return x, steps

DIRECT_METHODS = {
    "Gauss elimination": gauss_elimination,
    "Gauss-Jordan":      gauss_jordan,
    "LU":                lu_solve,
    "Cholesky":          cholesky_solve,
}

# ---- correctness check against a hand-verifiable 3x3 system -------------------
_A3 = np.array([[10., 1, 1], [1, 10, 1], [1, 1, 10]])
_b3 = np.array([12., 12, 12])          # exact answer is (1, 1, 1)
print("Verifying all four direct solvers on a hand-checkable 3x3 system")
print("   10x +  y +  z = 12")
print("    x + 10y +  z = 12   ->  true answer x = y = z = 1")
print("    x +  y + 10z = 12\n")
for _name, _fn in DIRECT_METHODS.items():
    _x, _st = _fn(_A3, _b3)
    _ok = np.allclose(_x, 1.0)
    print(f"  {_name:<20s} -> {np.round(_x, 12)}  steps={_st}  {'PASS' if _ok else 'FAIL'}")
    assert _ok, f"{_name} failed the 3x3 sanity check"
print("\n  All four direct solvers reproduce the exact answer.")

---
# 7 · The four iterative solvers

Iterative methods start from a guess and refine it. Each step is cheap — cost scales
with the number of **nonzeros**, not $n^2$ — so they stay fast on large sparse
matrices. The catch: **they are not guaranteed to work.** Whether the sequence
converges depends on the structure of $A$, which is precisely what the base paper
analyses theoretically and what this notebook tests empirically on real data.

| Method | Update rule, in words |
|---|---|
| **Jacobi** | Update every unknown using only *previous*-round values. |
| **Gauss-Seidel** | Same, but immediately reuse each fresh value within the same round. |
| **SOR** | Gauss-Seidel plus a relaxation factor $\omega$ that over-corrects each step. |
| **Conjugate Gradient** | Moves along mutually conjugate directions; needs a symmetric positive-definite matrix, but converges far faster when applicable. |

All four stop when the relative residual falls below `tolerance`, or give up after
`max_iterations` and are recorded as **did not converge** — a real, reportable result,
not an error.

In [ ]:
def _rel_residual(A, x, b, b_norm):
    return np.linalg.norm(A @ x - b) / b_norm

# If the residual climbs this far above its best-ever value the iteration is
# diverging: stop immediately instead of burning the full iteration budget.
# Separates "diverged" from "converging, just too slowly", and is what makes a
# sweep over hundreds of matrices affordable.
DIVERGENCE_GROWTH = 1e4

def jacobi(A, b, max_iter, tol):
    d = A.diagonal()
    if np.any(np.abs(d) < 1e-14):
        raise SolverNotApplicable("zero on the diagonal")
    bn = np.linalg.norm(b) or 1.0
    x = np.zeros(A.shape[0]); Dinv = 1.0 / d
    for k in range(1, max_iter + 1):
        x_new = x + Dinv * (b - A @ x)
        if not np.all(np.isfinite(x_new)):
            return x_new, k, False              # diverged to inf/nan
        rr = _rel_residual(A, x_new, b, bn)
        if rr < tol:
            return x_new, k, True
        if k == 1:
            best = rr
        elif rr > best * DIVERGENCE_GROWTH:
            return x_new, k, False          # residual blowing up: diverged
        else:
            best = min(best, rr)
        x = x_new
    return x, max_iter, False

def gauss_seidel(A, b, max_iter, tol):
    L = sp.tril(A, format="csr"); U = sp.triu(A, k=1, format="csr")
    if np.any(np.abs(L.diagonal()) < 1e-14):
        raise SolverNotApplicable("zero on the diagonal")
    bn = np.linalg.norm(b) or 1.0
    x = np.zeros(A.shape[0])
    for k in range(1, max_iter + 1):
        x_new = spla.spsolve_triangular(L, b - U @ x, lower=True)
        if not np.all(np.isfinite(x_new)):
            return x_new, k, False
        rr = _rel_residual(A, x_new, b, bn)
        if rr < tol:
            return x_new, k, True
        if k == 1:
            best = rr
        elif rr > best * DIVERGENCE_GROWTH:
            return x_new, k, False          # residual blowing up: diverged
        else:
            best = min(best, rr)
        x = x_new
    return x, max_iter, False

def sor(A, b, max_iter, tol, omega):
    d = A.diagonal()
    if np.any(np.abs(d) < 1e-14):
        raise SolverNotApplicable("zero on the diagonal")
    D = sp.diags(d, format="csr")
    L = sp.tril(A, k=-1, format="csr"); U = sp.triu(A, k=1, format="csr")
    lhs = (D + omega * L).tocsr()
    bn = np.linalg.norm(b) or 1.0
    x = np.zeros(A.shape[0])
    for k in range(1, max_iter + 1):
        rhs = omega * b - (omega * U + (omega - 1) * D) @ x
        x_new = spla.spsolve_triangular(lhs, rhs, lower=True)
        if not np.all(np.isfinite(x_new)):
            return x_new, k, False
        rr = _rel_residual(A, x_new, b, bn)
        if rr < tol:
            return x_new, k, True
        if k == 1:
            best = rr
        elif rr > best * DIVERGENCE_GROWTH:
            return x_new, k, False          # residual blowing up: diverged
        else:
            best = min(best, rr)
        x = x_new
    return x, max_iter, False

def conjugate_gradient(A, b, max_iter, tol, sym_tol=1e-6):
    diff = (A - A.T)
    if diff.nnz > 0 and np.abs(diff).max() > sym_tol:
        raise SolverNotApplicable("not symmetric")
    x = np.zeros(A.shape[0]); r = b - A @ x; p = r.copy()
    rs_old = r @ r; bn = np.linalg.norm(b) or 1.0
    if np.sqrt(rs_old) / bn < tol:
        return x, 0, True
    for k in range(1, max_iter + 1):
        Ap = A @ p
        denom = p @ Ap
        if denom <= 0:
            raise SolverNotApplicable("not positive-definite (non-positive curvature)")
        alpha = rs_old / denom
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = r @ r
        if not np.isfinite(rs_new):
            return x, k, False
        rr = np.sqrt(rs_new) / bn
        if rr < tol:
            return x, k, True
        if k == 1:
            best = rr
        elif rr > best * DIVERGENCE_GROWTH:
            return x, k, False              # residual blowing up: diverged
        else:
            best = min(best, rr)
        p = r + (rs_new / rs_old) * p
        rs_old = rs_new
    return x, max_iter, False

# =============================================================================
#  THIS PROJECT'S PROPOSED METHODS
# =============================================================================
# The four classical iterative methods above fail on most real matrices for two
# structural reasons, both visible in our own benchmark:
#   1. Jacobi / Gauss-Seidel / SOR need (near) diagonal dominance, which most
#      real engineering matrices do not have.
#   2. Conjugate Gradient is mathematically restricted to symmetric
#      positive-definite matrices, ruling out every unsymmetric system.
#
# We address both. Neither ingredient is our invention -- BiCGSTAB (van der
# Vorst, 1992) and ILU preconditioning are established -- and we say so plainly.
# What IS ours is the ADAPTIVE DISPATCH RULE in APK below: the decision logic
# and its thresholds are derived from this project's own 830-matrix sweep.

def bicgstab(A, b, max_iter, tol, M=None):
    """BiCGSTAB, hand-written. Unlike CG this needs no symmetry, so it applies
    to the unsymmetric matrices that dominate the circuit and CFD domains.
    M is an optional preconditioner exposing .solve(vec)."""
    n = A.shape[0]
    psolve = (lambda v: M.solve(v)) if M is not None else (lambda v: v)
    x = np.zeros(n)
    r = b - A @ x
    r_hat = r.copy()
    rho = alpha = omega = 1.0
    v = np.zeros(n); pvec = np.zeros(n)
    bn = np.linalg.norm(b) or 1.0
    best = np.inf

    for k in range(1, max_iter + 1):
        rho_new = r_hat @ r
        if abs(rho_new) < 1e-300:
            return x, k, False                      # breakdown
        if k == 1:
            pvec = r.copy()
        else:
            beta = (rho_new / rho) * (alpha / omega)
            pvec = r + beta * (pvec - omega * v)
        y = psolve(pvec)
        v = A @ y
        denom = r_hat @ v
        if abs(denom) < 1e-300:
            return x, k, False
        alpha = rho_new / denom
        sv = r - alpha * v
        if np.linalg.norm(sv) / bn < tol:
            x = x + alpha * y
            return x, k, True
        z = psolve(sv)
        t = A @ z
        tt = t @ t
        if tt < 1e-300:
            return x, k, False
        omega = (t @ sv) / tt
        x = x + alpha * y + omega * z
        r = sv - omega * t
        if not np.all(np.isfinite(x)):
            return x, k, False
        rr = np.linalg.norm(r) / bn
        if rr < tol:
            return x, k, True
        if rr > best * DIVERGENCE_GROWTH:
            return x, k, False
        best = min(best, rr)
        rho = rho_new
    return x, max_iter, False


def apk(A, b, max_iter, tol):
    """APK - Adaptive Preconditioned Krylov (this project's proposed solver).

    Dispatch rule, derived from our own benchmark results:
      * Build an ILU preconditioner, relaxing fill/drop tolerance on failure
        (severely ill-conditioned matrices reject aggressive factorisations).
      * Symmetric matrix  -> preconditioned CG, falling back to BiCGSTAB if the
        matrix turns out to be indefinite (CG needs positive-definiteness, and
        symmetry alone does not guarantee it -- a failure mode we hit repeatedly
        in the eigenvalue/model-reduction domains).
      * Unsymmetric matrix -> preconditioned BiCGSTAB.

    The ILU factorisation itself uses scipy's spilu; writing a competitive
    incomplete-LU from scratch is out of scope and we do not claim it.
    """
    n = A.shape[0]
    Acsc = A.tocsc()

    # --- ILU with conservative fill ------------------------------------------
    # Fill factor matters more than it looks. An earlier version of this solver
    # used drop_tol=1e-4, fill_factor=10 and was killed outright by the OS on
    # M40PI_n1 (exactly singular, 1,087 of 2,028 diagonal entries zero): SuperLU
    # chased fill-in until it exhausted memory, which kills the process rather
    # than raising. Measured across this dataset, conservative settings produce
    # fill of only 0.6x-4.4x AND turn that hard kill into a catchable
    # RuntimeError, so every failure below is recoverable.
    diag = A.diagonal()

    M = None
    for drop_tol, fill in [(1e-3, 5.0), (1e-2, 3.0), (1e-1, 2.0)]:
        try:
            M = spla.spilu(Acsc, drop_tol=drop_tol, fill_factor=fill)
            break
        except Exception:
            continue        # singular or failed factorisation: relax and retry
    # If ILU is unavailable (singular-ish diagonal, or every attempt failed) we
    # fall back to a Jacobi/diagonal preconditioner, which never fills in.
    if M is None:
        safe = np.where(np.abs(diag) < 1e-14, 1.0, diag)
        M = _DiagonalPreconditioner(safe)

    diff = (A - A.T)
    is_sym = (diff.nnz == 0) or (abs(diff).max() <= 1e-8 * max(abs(A).max(), 1.0))

    def _core(rhs):
        """Solve A z = rhs with the dispatch rule, returning (z, iters, ok)."""
        if is_sym:
            try:
                z, kk, good = _pcg(A, rhs, max_iter, tol, M)
                if good:
                    return z, kk, True
            except _Indefinite:
                pass        # symmetric but indefinite -> drop the SPD assumption
        return bicgstab(A, rhs, max_iter, tol, M)

    x, k, ok = _core(b)
    if not ok:
        return x, k, False

    # --- iterative refinement -------------------------------------------------
    # A Krylov method stops as soon as the relative residual clears `tol`, which
    # leaves accuracy pinned at roughly that tolerance. Refinement recovers the
    # lost digits cheaply: solve A dx = r for the current residual r and correct
    # x by dx. Each pass squares the error reduction, and because the ILU
    # factorisation is already built the extra solves are nearly free.
    # We keep a correction only if it actually reduces the residual, so
    # refinement can never make the answer worse than the unrefined one.
    bn = np.linalg.norm(b) or 1.0
    best_res = np.linalg.norm(b - A @ x) / bn
    for _ in range(CONFIG.get("refinement_passes", 2)):
        if best_res <= 1e-15:
            break                       # already at machine precision
        r = b - A @ x
        dx, k_ref, ok_ref = _core(r)
        if not ok_ref or not np.all(np.isfinite(dx)):
            break
        x_cand = x + dx
        res_cand = np.linalg.norm(b - A @ x_cand) / bn
        if not np.isfinite(res_cand) or res_cand >= best_res:
            break                       # no improvement: keep the better answer
        x, best_res, k = x_cand, res_cand, k + k_ref
    return x, k, True


class _DiagonalPreconditioner:
    """Jacobi preconditioner: M^-1 = diag(A)^-1. Cannot fill in, cannot blow up
    memory, and still helps whenever the diagonal carries real scale information."""
    __slots__ = ("_inv",)

    def __init__(self, diag_values):
        self._inv = 1.0 / diag_values

    def solve(self, v):
        return self._inv * v


class _Indefinite(Exception):
    """Raised when CG meets non-positive curvature (matrix is not SPD)."""


def _pcg(A, b, max_iter, tol, M):
    """Preconditioned Conjugate Gradient, hand-written."""
    n = A.shape[0]
    psolve = (lambda v: M.solve(v)) if M is not None else (lambda v: v)
    x = np.zeros(n)
    r = b - A @ x
    z = psolve(r)
    pvec = z.copy()
    rz = r @ z
    bn = np.linalg.norm(b) or 1.0
    best = np.inf
    for k in range(1, max_iter + 1):
        Ap = A @ pvec
        denom = pvec @ Ap
        if denom <= 0:
            raise _Indefinite()
        alpha = rz / denom
        x = x + alpha * pvec
        r = r - alpha * Ap
        rr = np.linalg.norm(r) / bn
        if not np.isfinite(rr):
            return x, k, False
        if rr < tol:
            return x, k, True
        if rr > best * DIVERGENCE_GROWTH:
            return x, k, False
        best = min(best, rr)
        z = psolve(r)
        rz_new = r @ z
        if abs(rz) < 1e-300:
            return x, k, False
        pvec = z + (rz_new / rz) * pvec
        rz = rz_new
    return x, max_iter, False


ITERATIVE_METHODS = {
    "Jacobi":             lambda A, b: jacobi(A, b, CONFIG["max_iterations"], CONFIG["tolerance"]),
    "Gauss-Seidel":       lambda A, b: gauss_seidel(A, b, CONFIG["max_iterations"], CONFIG["tolerance"]),
    "SOR":                lambda A, b: sor(A, b, CONFIG["max_iterations"], CONFIG["tolerance"], CONFIG["sor_omega"]),
    "Conjugate Gradient": lambda A, b: conjugate_gradient(A, b, CONFIG["max_iterations"], CONFIG["tolerance"]),
    "BiCGSTAB":           lambda A, b: bicgstab(A, b, CONFIG["max_iterations"], CONFIG["tolerance"]),
    "APK (ours)":         lambda A, b: apk(A, b, CONFIG["max_iterations"], CONFIG["tolerance"]),
}

# ---- sanity check on the same diagonally-dominant 3x3 -------------------------
_A3s = sp.csr_matrix(_A3)
print("Verifying all four iterative solvers on the same 3x3 system\n")
for _name, _fn in ITERATIVE_METHODS.items():
    _x, _its, _conv = _fn(_A3s, _b3)
    _ok = _conv and np.allclose(_x, 1.0, atol=1e-6)
    print(f"  {_name:<20s} -> {np.round(_x, 8)}  iters={_its:<5d} converged={_conv}  {'PASS' if _ok else 'FAIL'}")
    assert _ok, f"{_name} failed the 3x3 sanity check"
print("\n  All four iterative solvers converge to the exact answer.")
print("  Note Gauss-Seidel/SOR/CG need far fewer iterations than Jacobi — the expected ordering.")

---
# 8 · Condition number

The condition number $\kappa(A)$ answers: *how much does a small error in the input
get amplified in the answer?* Low (near 1) means well-behaved; high means
**ill-conditioned** and numerically fragile.

It is the variable we correlate everything else against — the expectation being that
as $\kappa$ climbs, direct methods lose accuracy and iterative methods need more
iterations or fail outright.

Computing it exactly needs an SVD ($O(n^3)$), impractical for the largest matrices
here. So: **exact SVD below `cond_exact_cap`, and a standard 1-norm estimator
(`splu` + `onenormest`) above it**, so every matrix still gets a usable number.
Which route was used is recorded alongside the value.

In [ ]:
def condition_number(A_sparse, A_dense, n):
    """Exact SVD-based cond for small n; 1-norm estimate (splu + onenormest) for large n."""
    if n <= CONFIG["cond_exact_cap"] and A_dense is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                return float(np.linalg.cond(A_dense)), "exact_svd"
            except Exception:
                return float("nan"), "failed"
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            Acsc = A_sparse.tocsc()
            lu = spla.splu(Acsc)
            norm_A = spla.onenormest(Acsc)
            op = spla.LinearOperator(A_sparse.shape, matvec=lu.solve,
                                     rmatvec=lambda v: lu.solve(v, "T"))
            return float(norm_A * spla.onenormest(op)), "estimate_1norm"
    except Exception:
        return float("nan"), "failed"

print("Condition-number routing:")
print(f"  n <= {CONFIG['cond_exact_cap']:,}  -> exact SVD")
print(f"  n >  {CONFIG['cond_exact_cap']:,}  -> 1-norm estimate (splu + onenormest)")

---
# 9 · The benchmark harness

For every (matrix × method) pair this records the five columns promised in the
proposal: **runtime, iterations, residual, error, condition number** — plus a
`status` field that distinguishes the three ways a run can end:

| Status | Meaning |
|---|---|
| `ok` | solved to tolerance |
| `did_not_converge` | iterative method hit the iteration cap — a genuine result |
| `not_applicable` | method's requirements unmet (e.g. Cholesky on a non-symmetric matrix) |
| `skipped_too_large` | direct method above the size cap |

In [ ]:
def run_one_matrix(domain, name, mtx_path, verbose=True):
    A = load_matrix(mtx_path)
    n = A.shape[0]

    # --- structural singularity screen ---------------------------------------
    # A matrix with an all-zero row or column is STRUCTURALLY SINGULAR: Ax=b has
    # either no solution or infinitely many, so no unique x exists and every
    # solver result on it would be meaningless. We record that fact and move on
    # rather than benchmarking against an undefined answer.
    #
    # This is also a hard robustness requirement, not just a correctness one.
    # SuperLU's behaviour on these differs by version: on scipy 1.17 (local) the
    # factorisation raises a catchable RuntimeError, but on scipy 1.16.3
    # (Kaggle's image) it instead runs away with memory and the OS kills the
    # whole process. Two runs died here, both at M40PI_n1 (1 zero row, 1 zero
    # column). 19 of 930 matrices in this collection are affected.
    zero_rows = int((np.diff(A.indptr) == 0).sum())
    zero_cols = int((np.diff(A.tocsc().indptr) == 0).sum())
    if zero_rows or zero_cols:
        if verbose:
            print(f"    STRUCTURALLY SINGULAR: {zero_rows} zero row(s), "
                  f"{zero_cols} zero column(s) -> no unique solution; skipping all solvers")
        base = {"domain": domain, "matrix": name, "n": n, "nnz": A.nnz,
                "density": A.nnz / (n * n), "condition_number": np.inf,
                "condition_method": "structurally_singular"}
        out = []
        for mname in list(DIRECT_METHODS) + list(ITERATIVE_METHODS):
            fam = "direct" if mname in DIRECT_METHODS else "iterative"
            out.append({**base, "method": mname, "type": fam,
                        "status": "structurally_singular", "runtime_sec": None,
                        "iterations": None, "residual_abs": None, "error_abs": None,
                        "residual_rel": None, "error_rel": None})
        return out

    x_true, b = make_ground_truth(A)
    b_norm = np.linalg.norm(b) or 1.0
    xt_norm = np.linalg.norm(x_true) or 1.0

    needs_dense = n <= max(CONFIG["direct_size_cap"], CONFIG["cond_exact_cap"])
    A_dense = A.toarray() if needs_dense else None

    t_cond = time.perf_counter()
    cond, cond_method = condition_number(A, A_dense, n)
    t_cond = time.perf_counter() - t_cond

    base = {"domain": domain, "matrix": name, "n": n, "nnz": A.nnz,
            "density": A.nnz / (n * n), "condition_number": cond,
            "condition_method": cond_method}
    if verbose:
        print(f"    n={n:,}  nnz={A.nnz:,}  cond={cond:.3e} ({cond_method}, {t_cond:.1f}s)")

    rows = []
    for mname, fn in DIRECT_METHODS.items():
        row = dict(base, method=mname, type="direct")
        if n > CONFIG["direct_size_cap"]:
            row.update(status="skipped_too_large", runtime_sec=np.nan, iterations=np.nan,
                       residual_rel=np.nan, error_rel=np.nan, residual_abs=np.nan, error_abs=np.nan)
        else:
            try:
                t0 = time.perf_counter(); x, steps = fn(A_dense, b); dt = time.perf_counter() - t0
                ra = float(np.linalg.norm(A @ x - b)); ea = float(np.linalg.norm(x - x_true))
                row.update(status="ok", runtime_sec=dt, iterations=steps,
                           residual_abs=ra, error_abs=ea,
                           residual_rel=ra / b_norm, error_rel=ea / xt_norm)
            except SolverNotApplicable as e:
                row.update(status="not_applicable", note=str(e), runtime_sec=np.nan,
                           iterations=np.nan, residual_rel=np.nan, error_rel=np.nan,
                           residual_abs=np.nan, error_abs=np.nan)
            except Exception as e:
                row.update(status="error", note=f"{type(e).__name__}: {e}", runtime_sec=np.nan,
                           iterations=np.nan, residual_rel=np.nan, error_rel=np.nan,
                           residual_abs=np.nan, error_abs=np.nan)
        rows.append(row)
        if verbose:
            print(f"      {mname:<20s} {row['status']}")

    for mname, fn in ITERATIVE_METHODS.items():
        row = dict(base, method=mname, type="iterative")
        try:
            t0 = time.perf_counter(); x, its, conv = fn(A, b); dt = time.perf_counter() - t0
            finite = np.all(np.isfinite(x))
            ra = float(np.linalg.norm(A @ x - b)) if finite else np.nan
            ea = float(np.linalg.norm(x - x_true)) if finite else np.nan
            row.update(status="ok" if conv else "did_not_converge", runtime_sec=dt,
                       iterations=its, residual_abs=ra, error_abs=ea,
                       residual_rel=ra / b_norm if finite else np.nan,
                       error_rel=ea / xt_norm if finite else np.nan)
        except SolverNotApplicable as e:
            row.update(status="not_applicable", note=str(e), runtime_sec=np.nan,
                       iterations=np.nan, residual_rel=np.nan, error_rel=np.nan,
                       residual_abs=np.nan, error_abs=np.nan)
        except Exception as e:
            row.update(status="error", note=f"{type(e).__name__}: {e}", runtime_sec=np.nan,
                       iterations=np.nan, residual_rel=np.nan, error_rel=np.nan,
                       residual_abs=np.nan, error_abs=np.nan)
        rows.append(row)
        if verbose:
            extra = f" ({int(row['iterations']):,} iters)" if np.isfinite(row.get("iterations", np.nan)) else ""
            print(f"      {mname:<20s} {row['status']}{extra}")
    return rows

print(f"Harness ready — {len(DIRECT_METHODS)+len(ITERATIVE_METHODS)} methods x {len(MATRICES)} matrices "
      f"= {(len(DIRECT_METHODS)+len(ITERATIVE_METHODS))*len(MATRICES)} measurements to collect.")

---
# 10 · Running the full sweep

This is the long cell. Progress is printed per matrix so you can follow along;
partial results are checkpointed to CSV after every matrix, so a timeout never
loses everything already computed.

In [ ]:
print("=" * 78)
print("STARTING FULL BENCHMARK SWEEP")
print("=" * 78)

all_rows, sweep_t0 = [], time.perf_counter()
checkpoint = TABLE_DIR / "03_benchmark_results.csv"
order = sorted(MATRICES, key=lambda e: manifest.set_index("name")["rows"].get(e["name"], 0))

for idx, entry in enumerate(order, 1):
    elapsed = time.perf_counter() - sweep_t0
    print(f"\n[{idx:2d}/{len(order)}] {entry['domain']}/{entry['name']}"
          f"   (elapsed {elapsed/60:.1f} min)")
    try:
        all_rows.extend(run_one_matrix(entry["domain"], entry["name"], entry["path"]))
    except Exception as e:
        print(f"    !! matrix-level failure: {type(e).__name__}: {e}")
        all_rows.append({"domain": entry["domain"], "matrix": entry["name"],
                         "method": "ALL", "type": "ALL", "status": "matrix_load_failed",
                         "note": str(e)})
    pd.DataFrame(all_rows).to_csv(checkpoint, index=False)   # checkpoint every matrix

results = pd.DataFrame(all_rows)
total_min = (time.perf_counter() - sweep_t0) / 60

print("\n" + "=" * 78)
print(f"SWEEP COMPLETE in {total_min:.1f} minutes")
print("=" * 78)
print(f"  rows collected : {len(results):,}")
print(f"  matrices       : {results['matrix'].nunique()}")
print("\n  Status breakdown:")
for status, cnt in results["status"].value_counts().items():
    print(f"    {status:<22s} {cnt:4d}")
results.to_csv(checkpoint, index=False)
print(f"\n  [table saved] {checkpoint.name}")

---
# 11 · Results

Every figure below is written to `figures/` and bundled into the final ZIP.

In [ ]:
ok = results[results["status"] == "ok"].copy()
present_methods = [m for m in METHOD_ORDER if m in set(results["method"])]

print(f"Successful solves: {len(ok):,} of {len(results):,} attempted")
print(f"Methods present  : {len(present_methods)}")

# ---------------------------------------------- FIGURE: outcome heatmap
STATUS_LEVELS = ["ok", "did_not_converge", "not_applicable", "skipped_too_large", "error"]
STATUS_COLORS = {"ok": C_GREEN, "did_not_converge": C_RED, "not_applicable": "#C9D2D6",
                 "skipped_too_large": "#EAEFF1", "error": "#7A2E2E"}
mats = (results.drop_duplicates("matrix").sort_values("n")["matrix"].tolist())
grid = np.full((len(mats), len(present_methods)), np.nan)
for i, m in enumerate(mats):
    for j, meth in enumerate(present_methods):
        s = results[(results["matrix"] == m) & (results["method"] == meth)]
        if len(s):
            st = s.iloc[0]["status"]
            grid[i, j] = STATUS_LEVELS.index(st) if st in STATUS_LEVELS else np.nan

fig, ax = plt.subplots(figsize=(9.5, 0.235 * len(mats) + 2.4))
cmap = matplotlib.colors.ListedColormap([STATUS_COLORS[s] for s in STATUS_LEVELS])
ax.imshow(grid, aspect="auto", cmap=cmap, vmin=-0.5, vmax=len(STATUS_LEVELS) - 0.5)
ax.set_xticks(range(len(present_methods)))
ax.set_xticklabels(present_methods, rotation=38, ha="right", fontsize=9)
ax.set_yticks(range(len(mats))); ax.set_yticklabels(mats, fontsize=6.2)
ax.set_xticks(np.arange(-.5, len(present_methods), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(mats), 1), minor=True)
ax.grid(which="minor", color=SURFACE, linewidth=1.4); ax.grid(which="major", visible=False)
ax.tick_params(which="minor", length=0)
ax.axvline(3.5, color=INK, linewidth=2.2)
ax.text(1.5, -1.6, "DIRECT", ha="center", fontsize=9, fontweight="bold", color=C_TEAL)
ax.text(5.5, -1.6, "ITERATIVE", ha="center", fontsize=9, fontweight="bold", color=C_RED)
ax.legend(handles=[matplotlib.patches.Patch(facecolor=STATUS_COLORS[s], label=s.replace("_", " "))
                   for s in STATUS_LEVELS],
          loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8.5)
ax.set_title("Which solver works on which matrix", loc="left", pad=26)
fig.tight_layout()
save_fig(fig, "03_outcome_matrix", "Outcome of every (matrix, method) pair. Rows sorted by matrix size.")
plt.show()

In [ ]:
# ---------------------------------------------- FIGURE: accuracy heatmap
acc = np.full((len(mats), len(present_methods)), np.nan)
for i, m in enumerate(mats):
    for j, meth in enumerate(present_methods):
        s = ok[(ok["matrix"] == m) & (ok["method"] == meth)]
        if len(s) and np.isfinite(s.iloc[0]["error_rel"]):
            acc[i, j] = np.log10(max(s.iloc[0]["error_rel"], 1e-18))

fig, ax = plt.subplots(figsize=(9.5, 0.235 * len(mats) + 2.4))
im = ax.imshow(acc, aspect="auto", cmap=SEQ_CMAP)
ax.set_xticks(range(len(present_methods)))
ax.set_xticklabels(present_methods, rotation=38, ha="right", fontsize=9)
ax.set_yticks(range(len(mats))); ax.set_yticklabels(mats, fontsize=6.2)
ax.set_xticks(np.arange(-.5, len(present_methods), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(mats), 1), minor=True)
ax.grid(which="minor", color=SURFACE, linewidth=1.4); ax.grid(which="major", visible=False)
ax.tick_params(which="minor", length=0)
ax.axvline(3.5, color=INK, linewidth=2.2)
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label("$\\log_{10}$ relative error   (lower = more accurate)", fontsize=9)
ax.set_title("Accuracy achieved, where the method applied", loc="left", pad=12)
fig.text(0.5, -0.02, "blank cells = method not applicable, diverged, or skipped",
         fontsize=8, color=INK_MUTED, ha="center")
fig.tight_layout()
save_fig(fig, "04_accuracy_heatmap", "Relative error per (matrix, method); darker means more accurate.")
plt.show()

In [ ]:
# ---------------------------------------------- FIGURE: runtime vs size
fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True)
for ax, fam in zip(axes, ["direct", "iterative"]):
    sub_all = ok[ok["type"] == fam]
    for meth in [m for m in present_methods if METHOD_FAMILY.get(m) == fam]:
        s = sub_all[sub_all["method"] == meth].sort_values("n")
        if not len(s):
            continue
        ax.plot(s["n"], s["runtime_sec"], marker=METHOD_MARKERS[meth], markersize=5.5,
                linewidth=1.6, color=FAMILY_COLORS[fam], alpha=0.85,
                markeredgecolor=SURFACE, markeredgewidth=0.7, label=meth)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("matrix size $n$")
    ax.set_title(f"{fam.title()} methods", loc="left", color=FAMILY_COLORS[fam])
    ax.legend(fontsize=8.5, loc="upper left")
axes[0].set_ylabel("runtime (seconds)")

# O(n^3) reference slope on the direct panel
_d = ok[ok["type"] == "direct"]
if len(_d):
    n0 = _d["n"].min(); t0 = _d[_d["n"] == n0]["runtime_sec"].median()
    ns = np.array(sorted(_d["n"].unique()), dtype=float)
    axes[0].plot(ns, t0 * (ns / n0) ** 3, linestyle=":", color=INK_MUTED, linewidth=1.5, zorder=0)
    axes[0].text(ns[-1], t0 * (ns[-1] / n0) ** 3, "  $O(n^3)$", fontsize=9,
                 color=INK_MUTED, va="center")

fig.suptitle("Runtime scaling — direct methods pay $O(n^3)$, iterative methods track sparsity",
             fontsize=13, fontweight="bold", x=0.005, ha="left", y=1.03, color=INK)
fig.tight_layout()
save_fig(fig, "05_runtime_vs_size", "Runtime against matrix size for both method families.")
plt.show()

In [ ]:
# ---------------------------------------------- FIGURE: iterations, direct vs iterative
fig, ax = plt.subplots(figsize=(9.6, 5.2))
it = ok[np.isfinite(ok["iterations"])]
for fam in ["direct", "iterative"]:
    for meth in [m for m in present_methods if METHOD_FAMILY.get(m) == fam]:
        s = it[it["method"] == meth].sort_values("n")
        if not len(s):
            continue
        ax.scatter(s["n"], s["iterations"], marker=METHOD_MARKERS[meth], s=46,
                   color=FAMILY_COLORS[fam], alpha=0.8, edgecolor=SURFACE,
                   linewidth=0.8, label=f"{meth} ({fam})")
ns = np.array(sorted(it["n"].unique()), dtype=float)
ax.plot(ns, ns, linestyle=":", color=INK_MUTED, linewidth=1.6, zorder=0)
ax.text(ns[-1], ns[-1], "  steps = $n$", fontsize=9, color=INK_MUTED, va="center")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("matrix size $n$"); ax.set_ylabel("steps / iterations taken")
ax.set_title("Step count: fixed by size for direct methods, data-dependent for iterative ones",
             loc="left")
ax.legend(fontsize=8, ncol=2, loc="upper left")
fig.tight_layout()
save_fig(fig, "06_iterations_direct_vs_iterative",
         "Direct methods sit exactly on steps=n; iterative counts scatter by matrix behaviour.")
plt.show()

In [ ]:
# ---------------------------------------------- FIGURE: conditioning vs accuracy
fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0))
cnd = ok[np.isfinite(ok["condition_number"]) & np.isfinite(ok["error_rel"])]

ax = axes[0]
for fam in ["direct", "iterative"]:
    s = cnd[cnd["type"] == fam]
    ax.scatter(s["condition_number"], s["error_rel"].clip(lower=1e-18), s=42,
               color=FAMILY_COLORS[fam], alpha=0.72, edgecolor=SURFACE, linewidth=0.7,
               label=f"{fam} methods")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"condition number $\kappa(A)$"); ax.set_ylabel("relative error")
ax.set_title("Ill-conditioning degrades accuracy", loc="left")
ax.legend(fontsize=9)

ax = axes[1]
cnv = results[results["type"] == "iterative"].copy()
cnv = cnv[np.isfinite(cnv["condition_number"])]
cnv["converged"] = (cnv["status"] == "ok").astype(int)
if len(cnv):
    cnv["kappa_bin"] = pd.cut(np.log10(cnv["condition_number"].clip(lower=1)),
                              bins=[0, 3, 6, 9, 12, 30],
                              labels=["<1e3", "1e3-1e6", "1e6-1e9", "1e9-1e12", ">1e12"])
    rate = cnv.groupby("kappa_bin", observed=True)["converged"].agg(["mean", "count"]).reset_index()
    bars = ax.bar(rate["kappa_bin"].astype(str), rate["mean"] * 100, color=C_TEAL, width=0.6)
    for bar, (_, r) in zip(bars, rate.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f"{r['mean']*100:.0f}%\n(n={int(r['count'])})", ha="center",
                fontsize=8.5, color=INK)
    ax.set_ylim(0, 112)
ax.set_xlabel(r"condition number $\kappa(A)$"); ax.set_ylabel("iterative runs that converged (%)")
ax.set_title("Convergence collapses as conditioning worsens", loc="left")
ax.grid(axis="x", visible=False)

fig.suptitle("Conditioning is the variable that predicts solver trouble",
             fontsize=13, fontweight="bold", x=0.005, ha="left", y=1.03, color=INK)
fig.tight_layout()
save_fig(fig, "07_conditioning_analysis", "Accuracy and convergence both degrade with condition number.")
plt.show()

In [ ]:
# ---------------------------------------------- FIGURE: Jacobi vs Gauss-Seidel (base-paper test)
js = results[results["method"].isin(["Jacobi", "Gauss-Seidel"])]
piv = js.pivot_table(index="matrix", columns="method", values="status", aggfunc="first").dropna()
if not {"Jacobi", "Gauss-Seidel"}.issubset(piv.columns):
    piv = pd.DataFrame({"Jacobi": [], "Gauss-Seidel": []})
both  = int(((piv["Jacobi"] == "ok") & (piv["Gauss-Seidel"] == "ok")).sum())
gs_on = int(((piv["Jacobi"] != "ok") & (piv["Gauss-Seidel"] == "ok")).sum())
jc_on = int(((piv["Jacobi"] == "ok") & (piv["Gauss-Seidel"] != "ok")).sum())
neither = int(((piv["Jacobi"] != "ok") & (piv["Gauss-Seidel"] != "ok")).sum())

fig, axes = plt.subplots(1, 2, figsize=(13.2, 4.8))
ax = axes[0]
cats = ["Both\nconverge", "Only\nGauss-Seidel", "Only\nJacobi", "Neither"]
vals = [both, gs_on, jc_on, neither]
cols = [C_GREEN, C_TEAL, C_GOLD, "#C9D2D6"]
bars = ax.bar(cats, vals, color=cols, width=0.62)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.02, str(v),
            ha="center", fontsize=11, fontweight="bold", color=INK)
ax.set_ylabel("number of matrices"); ax.set_ylim(0, max(vals) * 1.2)
ax.set_title("Jacobi vs. Gauss-Seidel on real matrices", loc="left")
ax.grid(axis="x", visible=False)

ax = axes[1]
pair = ok[ok["method"].isin(["Jacobi", "Gauss-Seidel"])]
pv = pair.pivot_table(index="matrix", columns="method", values="iterations", aggfunc="first").dropna()
if len(pv) and {"Jacobi", "Gauss-Seidel"}.issubset(pv.columns):
    ax.scatter(pv["Jacobi"], pv["Gauss-Seidel"], s=54, color=C_TEAL,
               edgecolor=SURFACE, linewidth=0.9, alpha=0.9)
    lim = [min(pv.min()) * 0.6, max(pv.max()) * 1.6]
    ax.plot(lim, lim, linestyle=":", color=INK_MUTED, linewidth=1.5)
    ax.text(lim[1], lim[1], "  equal", fontsize=8.5, color=INK_MUTED, va="center")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lim); ax.set_ylim(lim)
    faster = int((pv["Gauss-Seidel"] < pv["Jacobi"]).sum())
    ax.text(0.04, 0.94, f"Gauss-Seidel needed fewer iterations\non {faster} of {len(pv)} shared matrices",
            transform=ax.transAxes, fontsize=9, color=INK, va="top")
ax.set_xlabel("Jacobi iterations"); ax.set_ylabel("Gauss-Seidel iterations")
ax.set_title("Head-to-head iteration count", loc="left")

fig.suptitle("Extending the base paper's headline comparison to real engineering matrices",
             fontsize=13, fontweight="bold", x=0.005, ha="left", y=1.04, color=INK)
fig.tight_layout()
save_fig(fig, "08_jacobi_vs_gauss_seidel",
         "The base paper's Jacobi/Gauss-Seidel comparison, re-run on real data.")
plt.show()

print(f"\n  Both converge      : {both}")
print(f"  Only Gauss-Seidel  : {gs_on}")
print(f"  Only Jacobi        : {jc_on}")
print(f"  Neither            : {neither}")

In [ ]:
# ---------------------------------------------- FIGURE: per-method scoreboard
# Reliability counts only matrices a method was actually able to ATTEMPT.
# "skipped_too_large" (direct methods on huge matrices, a cost decision) and
# "structurally_singular" (no unique solution exists) are not solver failures,
# so charging them against a method would misstate its reliability.
_NOT_ATTEMPTED = {"skipped_too_large", "structurally_singular", "matrix_load_failed"}
_att = results[~results["status"].isin(_NOT_ATTEMPTED)]
rate = (_att.assign(success=(_att["status"] == "ok").astype(int))
        .groupby("method")["success"].agg(["sum", "count"]))
rate["pct"] = rate["sum"] / rate["count"] * 100
rate = rate.reindex([m for m in present_methods if m in rate.index])

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.6))

ax = axes[0]
cols = [FAMILY_COLORS[METHOD_FAMILY[m]] for m in rate.index]
bars = ax.barh(list(rate.index), rate["pct"], color=cols, height=0.6)
for bar, (_, r) in zip(bars, rate.iterrows()):
    ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2,
            f"{r['pct']:.0f}%", va="center", fontsize=9, fontweight="bold", color=INK)
ax.set_xlim(0, 116); ax.set_xlabel("solved, % of matrices the method could attempt")
ax.set_title("Reliability", loc="left"); ax.grid(axis="y", visible=False); ax.invert_yaxis()

ax = axes[1]
med = ok.groupby("method")["error_rel"].median().reindex(rate.index).clip(lower=1e-18)
bars = ax.barh(list(med.index), med.values, color=[FAMILY_COLORS[METHOD_FAMILY[m]] for m in med.index], height=0.6)
ax.set_xscale("log"); ax.set_xlabel("median relative error (lower is better)")
ax.set_title("Accuracy", loc="left"); ax.grid(axis="y", visible=False); ax.invert_yaxis()

ax = axes[2]
mrt = ok.groupby("method")["runtime_sec"].median().reindex(rate.index)
ax.barh(list(mrt.index), mrt.values, color=[FAMILY_COLORS[METHOD_FAMILY[m]] for m in mrt.index], height=0.6)
ax.set_xscale("log"); ax.set_xlabel("median runtime, seconds (lower is better)")
ax.set_title("Speed", loc="left"); ax.grid(axis="y", visible=False); ax.invert_yaxis()

fig.legend(handles=[Line2D([0], [0], marker="s", linestyle="", markersize=9,
                           markerfacecolor=FAMILY_COLORS[f], markeredgecolor="none",
                           label=f"{f} methods") for f in ["direct", "iterative"]],
           loc="upper right", bbox_to_anchor=(0.995, 1.06), ncol=2, fontsize=9)
fig.suptitle(f"Solver scoreboard across all {results['matrix'].nunique()} matrices", fontsize=13, fontweight="bold",
             x=0.005, ha="left", y=1.05, color=INK)
fig.tight_layout()
save_fig(fig, "09_solver_scoreboard", "Reliability, accuracy and speed for all ten solvers, including our APK.")
plt.show()

## Paired accuracy: comparing solvers fairly

The scoreboard's accuracy panel compares each method's median error over **its own**
set of solved matrices, and that flatters the weak solvers: Jacobi's median covers
only the handful of easy systems it can handle at all, while APK's spans hundreds
including far harder ones. The honest comparison is **paired** -- restrict to
matrices *both* methods solved, then compare head to head.

In [ ]:
# ---------------------------------------------- FIGURE: paired accuracy vs APK
ok_ = results[results["status"] == "ok"]
piv_err = ok_.pivot_table(index="matrix", columns="method", values="error_rel", aggfunc="first")
have = set(piv_err.columns)
rivals = ([m for m in ITERATIVE_METHODS_ORDER if m != "APK (ours)" and m in have]
          if "APK (ours)" in have else [])

pair_rows = []
for m in rivals:
    common = piv_err[["APK (ours)", m]].dropna()
    if len(common) < 3:
        continue
    pair_rows.append({"rival": m, "common": len(common),
                      "apk_wins": int((common["APK (ours)"] < common[m]).sum()),
                      "apk_med": common["APK (ours)"].median(),
                      "rival_med": common[m].median()})
pair = pd.DataFrame(pair_rows)
if len(pair) == 0:
    print("Not enough overlapping solves for a paired comparison "
          "(expected only on a reduced test subset).")
else:
  pair["apk_win_pct"] = (pair["apk_wins"] / pair["common"] * 100).round(1)

  fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))

  ax = axes[0]
  bars = ax.barh(pair["rival"], pair["apk_win_pct"], color=C_TEAL, height=0.58)
  ax.axvline(50, color=INK_MUTED, linestyle="--", linewidth=1.1)
  ax.text(50.8, -0.42, "50% = tie", fontsize=7.5, color=INK_MUTED)
  for bar, (_, r) in zip(bars, pair.iterrows()):
      ax.text(bar.get_width() + 1.4, bar.get_y() + bar.get_height()/2,
              f"{r['apk_win_pct']:.0f}%  (n={r['common']})", va="center",
              fontsize=8.5, fontweight="bold", color=INK)
  ax.set_xlim(0, 118)
  ax.set_xlabel("matrices where APK is MORE accurate (%)")
  ax.set_title("APK vs each rival, on matrices both solved", loc="left")
  ax.grid(axis="y", visible=False); ax.invert_yaxis()

  ax = axes[1]
  idx = np.arange(len(pair)); w = 0.38
  ax.barh(idx - w/2, pair["apk_med"], height=w, color=C_TEAL, label="APK (ours)")
  ax.barh(idx + w/2, pair["rival_med"], height=w, color=C_RED, label="rival")
  ax.set_yticks(idx); ax.set_yticklabels(pair["rival"])
  ax.set_xscale("log"); ax.set_xlabel("median relative error on shared matrices (lower is better)")
  ax.set_title("Median error, like-for-like", loc="left")
  ax.legend(frameon=False, fontsize=8.5); ax.grid(axis="y", visible=False); ax.invert_yaxis()

  fig.suptitle("Paired accuracy comparison -- the like-for-like view",
               fontsize=13, fontweight="bold", color=INK, x=0.012, ha="left")
  fig.tight_layout(rect=[0, 0, 1, 0.93])
  save_fig(fig, "12_paired_accuracy", "APK vs each rival on matrices both solved.")
  plt.show()

  print("Paired accuracy (matrices BOTH methods solved):")
  for _, r in pair.iterrows():
      print(f"  vs {r['rival']:<20s} n={r['common']:>4d}  APK more accurate on "
            f"{r['apk_win_pct']:>5.1f}%   median {r['apk_med']:.2e} vs {r['rival_med']:.2e}")

In [ ]:
# ---------------------------------------------- FIGURE: per-domain reliability
dom_rate = (results.assign(success=(results["status"] == "ok").astype(int))
            .groupby(["domain", "method"])["success"].mean().unstack() * 100)
dom_rate = dom_rate.reindex(columns=[m for m in present_methods if m in dom_rate.columns])

fig, ax = plt.subplots(figsize=(10.5, 3.6))
im = ax.imshow(dom_rate.values, aspect="auto", cmap=SEQ_CMAP, vmin=0, vmax=100)
ax.set_xticks(range(len(dom_rate.columns)))
ax.set_xticklabels(dom_rate.columns, rotation=32, ha="right", fontsize=9)
ax.set_yticks(range(len(dom_rate.index)))
ax.set_yticklabels([d.replace("_", " ").title() for d in dom_rate.index], fontsize=10)
for i in range(dom_rate.shape[0]):
    for j in range(dom_rate.shape[1]):
        v = dom_rate.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=9,
                    color="white" if v > 55 else INK, fontweight="bold")
ax.set_xticks(np.arange(-.5, len(dom_rate.columns), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(dom_rate.index), 1), minor=True)
ax.grid(which="minor", color=SURFACE, linewidth=2); ax.grid(which="major", visible=False)
ax.tick_params(which="minor", length=0)
ax.axvline(3.5, color=INK, linewidth=2.2)
fig.colorbar(im, ax=ax, pad=0.02, label="success rate (%)")
ax.set_title("Which solver to trust, by engineering domain", loc="left", pad=12)
fig.tight_layout()
save_fig(fig, "10_domain_method_matrix", "Per-domain success rate for each solver.")
plt.show()

dom_rate.to_csv(TABLE_DIR / "04_success_rate_by_domain.csv")
print("  [table saved] 04_success_rate_by_domain.csv")

In [ ]:
# ---------------------------------------------- FIGURE: residual vs error
fig, ax = plt.subplots(figsize=(8.2, 5.6))
re_ = ok[np.isfinite(ok["residual_rel"]) & np.isfinite(ok["error_rel"])]
for fam in ["direct", "iterative"]:
    s = re_[re_["type"] == fam]
    ax.scatter(s["residual_rel"].clip(lower=1e-18), s["error_rel"].clip(lower=1e-18),
               s=46, color=FAMILY_COLORS[fam], alpha=0.72, edgecolor=SURFACE,
               linewidth=0.7, label=f"{fam} methods")
lims = [1e-18, 1e2]
ax.plot(lims, lims, linestyle=":", color=INK_MUTED, linewidth=1.5, zorder=0)
ax.text(lims[1], lims[1], "  equal", fontsize=8.5, color=INK_MUTED, va="center")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("relative residual  (is the equation satisfied?)")
ax.set_ylabel("relative error  (is the answer right?)")
ax.set_title("A small residual does not guarantee a correct answer", loc="left")
ax.text(0.03, 0.95, "points above the line:\nresidual looks fine,\nanswer is worse",
        transform=ax.transAxes, fontsize=8.8, color=INK_MUTED, va="top")
ax.legend(fontsize=9, loc="lower right")
fig.tight_layout()
save_fig(fig, "11_residual_vs_error", "Residual and error diverge most on ill-conditioned systems.")
plt.show()

---
# 12 · The decision guide

The actual deliverable promised in the proposal: a concrete, evidence-based answer
to *"which solver, for which real system"*.

In [ ]:
lines = ["# SolveBench — Solver Decision Guide\n\n",
         f"_Generated {datetime.now():%Y-%m-%d %H:%M} from {results['matrix'].nunique()} "
         f"real matrices x {len(present_methods)} solvers._\n\n",
         "## Overall\n\n"]

rel = (results.assign(s=(results["status"] == "ok").astype(int))
       .groupby("method")["s"].mean().sort_values(ascending=False) * 100)
acc_rank = ok.groupby("method")["error_rel"].median().sort_values()
spd_rank = ok.groupby("method")["runtime_sec"].median().sort_values()

lines.append(f"- **Most reliable:** {rel.index[0]} — solved {rel.iloc[0]:.0f}% of all matrices\n")
lines.append(f"- **Most accurate:** {acc_rank.index[0]} — median relative error {acc_rank.iloc[0]:.2e}\n")
lines.append(f"- **Fastest:** {spd_rank.index[0]} — median runtime {spd_rank.iloc[0]:.4f} s\n\n")
lines.append("| Method | Family | Success rate | Median rel. error | Median runtime (s) |\n")
lines.append("|---|---|---|---|---|\n")
for m in present_methods:
    lines.append(f"| {m} | {METHOD_FAMILY.get(m,'-')} | {rel.get(m, float('nan')):.0f}% | "
                 f"{acc_rank.get(m, float('nan')):.2e} | {spd_rank.get(m, float('nan')):.4f} |\n")

lines.append("\n## By domain\n")
for dom in sorted(results["domain"].unique()):
    if dom == "ALL":
        continue
    d_all, d_ok = results[results["domain"] == dom], ok[ok["domain"] == dom]
    lines.append(f"\n### {dom.replace('_',' ').title()}\n\n")
    if not len(d_ok):
        lines.append("_No successful solves recorded._\n"); continue
    d_rel = (d_all.assign(s=(d_all["status"] == "ok").astype(int))
             .groupby("method")["s"].mean().sort_values(ascending=False) * 100)
    d_acc = d_ok.groupby("method")["error_rel"].median().sort_values()
    d_spd = d_ok.groupby("method")["runtime_sec"].median().sort_values()
    lines.append(f"- **Recommended:** `{d_acc.index[0]}` (most accurate here, "
                 f"median relative error {d_acc.iloc[0]:.2e})\n")
    lines.append(f"- **Fastest:** `{d_spd.index[0]}` ({d_spd.iloc[0]:.4f} s median)\n")
    lines.append(f"- **Most reliable:** `{d_rel.index[0]}` ({d_rel.iloc[0]:.0f}% success)\n")
    failed = d_all[d_all["status"] == "did_not_converge"]["method"].unique()
    if len(failed):
        lines.append(f"- **Avoid / use with care:** {', '.join(sorted(failed))} "
                     f"(failed to converge on at least one matrix here)\n")

guide = "".join(lines)
(REPORT_DIR / "decision_guide.md").write_text(guide, encoding="utf-8")
print(guide)
print("\n  [report saved] decision_guide.md")

In [ ]:
# ---------------------------------------------- summary report
n_mat = results["matrix"].nunique()
summary_md = f"""# SolveBench — Run Summary

**Generated:** {datetime.now():%Y-%m-%d %H:%M:%S}
**Runtime:** {total_min:.1f} minutes
**Environment:** {'Kaggle' if ON_KAGGLE else 'local'} · Python {sys.version.split()[0]} · {os.cpu_count()} CPU cores

## Scope

| | |
|---|---|
| Matrices benchmarked | {n_mat} |
| Domains | {results['domain'].nunique()} |
| Solvers | {len(present_methods)} (4 direct, 4 iterative) |
| Total measurements | {len(results):,} |
| Size range | n = {int(results['n'].min()):,} to {int(results['n'].max()):,} |
| Successful solves | {len(ok):,} ({len(ok)/len(results)*100:.1f}%) |

## Outcome breakdown

| Status | Count |
|---|---|
""" + "".join(f"| {s} | {c} |\n" for s, c in results["status"].value_counts().items()) + f"""

## Key findings

1. **Jacobi vs Gauss-Seidel on real data** — both converge on {both} matrices;
   Gauss-Seidel alone on {gs_on}; Jacobi alone on {jc_on}; neither on {neither}.
   The base paper's claim that Gauss-Seidel usually (but not always) wins is
   tested here at a scale the paper never reached.
2. **Step count** — direct methods always take exactly n or n-1 stages, fixed by
   size alone; iterative counts vary by orders of magnitude on the same-size matrix
   depending on conditioning.
3. **Residual is not error** — see `11_residual_vs_error.png`; on ill-conditioned
   systems a solver can satisfy the equation while still being far from the answer.

## Output layout

```
solvebench_output/
  figures/   {len(SAVED_FIGURES)} PNG figures
  tables/    CSV result tables
  reports/   decision_guide.md, summary_report.md
  logs/      run configuration
```

## Figure index

| File | What it shows |
|---|---|
""" + "".join(f"| `{f['file']}` | {f['caption']} |\n" for f in SAVED_FIGURES)

(REPORT_DIR / "summary_report.md").write_text(summary_md, encoding="utf-8")
print(summary_md)

json.dump({"config": CONFIG, "started": str(RUN_STARTED), "runtime_minutes": total_min,
           "on_kaggle": ON_KAGGLE, "n_matrices": int(n_mat), "n_rows": int(len(results)),
           "figures": SAVED_FIGURES},
          open(LOG_DIR / "run_metadata.json", "w"), indent=2)
print("\n  [report saved] summary_report.md")
print("  [log saved] run_metadata.json")

---
# 13 · Packaging everything for download

Bundles figures, tables, and reports into a single ZIP. On Kaggle it lands in
`/kaggle/working/` — download it from the notebook's **Output** tab.

In [ ]:
# aggregate per-method table for convenience
agg = (results.assign(success=(results["status"] == "ok").astype(int))
       .groupby(["method", "type"])
       .agg(success_rate=("success", "mean"),
            median_runtime_s=("runtime_sec", "median"),
            median_rel_error=("error_rel", "median"),
            median_iterations=("iterations", "median"))
       .reset_index().sort_values(["type", "success_rate"], ascending=[True, False]))
agg["success_rate"] = (agg["success_rate"] * 100).round(1)
agg.to_csv(TABLE_DIR / "05_method_summary.csv", index=False)
print(agg.to_string(index=False))

readme = f"""# SolveBench results bundle

Generated {datetime.now():%Y-%m-%d %H:%M}

- `figures/`  — every chart, 160 dpi PNG, ready to drop into slides or a report
- `tables/`   — raw and aggregated results as CSV
- `reports/`  — decision guide and run summary (Markdown)
- `logs/`     — exact configuration used for this run

Start with `reports/summary_report.md`.
"""
(OUTPUT_ROOT / "README.md").write_text(readme, encoding="utf-8")

zip_base = (Path("/kaggle/working") if ON_KAGGLE else Path("results")) / "solvebench_results"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_ROOT)

print("\n" + "=" * 78)
print("OUTPUT BUNDLE READY")
print("=" * 78)
total = 0
for sub in ["figures", "tables", "reports", "logs"]:
    files = sorted((OUTPUT_ROOT / sub).glob("*"))
    total += len(files)
    print(f"\n  {sub}/  ({len(files)} files)")
    for f in files:
        print(f"      {f.name:<45s} {f.stat().st_size/1024:>8.1f} KB")
print(f"\n  {total} files total")
print(f"  ZIP: {zip_path}  ({Path(zip_path).stat().st_size/1024/1024:.2f} MB)")
print("\n  On Kaggle: Output tab -> download solvebench_results.zip")
print("=" * 78)

---

## Done

Everything above was produced from real SuiteSparse matrices with no synthetic data
and no randomness — re-running this notebook reproduces identical numbers.

**Next steps for the team**

* Raise `direct_size_cap` if you want the hand-written direct solvers to cover the
  largest matrices too (costs $O(n^3)$ time — see the runtime figure for the scaling).
* Tune `sor_omega`; the theoretically optimal $\omega$ differs per matrix, and
  sweeping it is a natural extension.
* Add a preconditioner to CG — usually the single biggest practical win available here.